# Part 3: Creating the model and evaluation metrics
Here we will create our models with data from parts 1 and 2.

In [1]:
#Imports
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#Our dataframe
final_df = pd.read_csv("cleaned_data/cleaned_df_final_test.csv") #TEMP NAME, remember to change.
#Variables that we need

## 3.1 Splitting training data for evaluation
To get some metrics of model performance, we will split our data so that we can verify the model is working well.

In [2]:
#Define what proportion should be reserved for verifying. Evaluation will be done with the latter part.
SPLIT_RATIO = 0.975

#Find what day this corresponds to
split_day_numerical = final_df["days_normalized"].quantile(SPLIT_RATIO)
print("Splitting at ", SPLIT_RATIO, ", Corresponds to day", split_day_numerical)

#Split data according to our index
train_df = final_df[final_df["days_normalized"] <= split_day_numerical]
test_df = final_df[final_df["days_normalized"] > split_day_numerical]
#Verify the split

print(f"Train range: {train_df['days_normalized'].min()} to {train_df['days_normalized'].max()}")
print(f"Test range: {test_df['days_normalized'].min()} to {test_df['days_normalized'].max()}")

first_test_date = test_df["date_str"].iloc[0]
last_test_date = test_df["date_str"].iloc[-1]
print("The data used for testings starts at ", first_test_date, ", and ends at ", last_test_date)



Splitting at  0.975 , Corresponds to day 7305.0
Train range: 0 to 7305
Test range: 7306 to 7492
The data used for testings starts at  2024-06-16 , and ends at  2024-12-19


## 3.2 Initializing the model
Here we initialize our models and select what columns will be used

In [ ]:
xgb_model = xgb.XGBRegressor()

#Select columns to use:
xgb_X_columns = ["day_of_week","is_weekend","month","is_holiday","rolling_7","rolling_30","rolling_100"]

## 3.3 Run test model with train and test split.


In [4]:
train_X = train_df[xgb_X_columns]
train_y = train_df["daily_weight"]
test_y = test_df["daily_weight"]

xgb_model.fit(train_X, train_y)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [58]:
#For test_data we can use what we have.

all_predictions = []
valid_ids = final_df["rm_id"].unique()

for id, group in test_df.groupby("rm_id", sort=False):
    group = group.sort_values('days_normalized')
    X_group = group[xgb_X_columns]
    y_pred = xgb_model.predict(X_group)
    group["predicted_weight"] = y_pred
    group["rm_id"] = id
    all_predictions.append(group)



### 3.3 test performances.
For making adjustments we can display some performance metrics now, but we will need to evaluate performance in next section (post-prediction-cleanup)

In [ ]:
#

prediction_df = pd.concat(all_predictions)
#prediction_df.sortby("days")
prediction_df["rm_id"] = pd.Categorical(prediction_df["rm_id"], categories=valid_ids, ordered=False)
y_pred = prediction_df["predicted_weight"]


mse = mean_squared_error(test_y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_y, y_pred)
r2 = r2_score(test_y, y_pred)


print(f'Mean squared Error: {mse:.4f}')
print(f'Root Mean Squared: {rmse:.4f}')
print(f'Mean absolute error: {mae:.4f}')
print(f'r2-score: {r2:.4f}')


#Best so far:
""" Mean squared Error: 18866094.3280
Root Mean Squared: 4343.5118
Mean absolute error: 793.8375
r2-score: 0.7197 """



Mean squared Error: 18866094.3280
Root Mean Squared: 4343.5118
Mean absolute error: 793.8375
r2-score: 0.7197


# 3.4 Create dataframe for prediction window
test split already has data such as day_of_week, month, holidays etc. We must prepare our prediction window (from 1st Jan 2025 to 31st May) to also hold these values so they can be used for prediction.

In [ ]:


train_X_full = final_df[xgb_X_columns]
train_y_full = final_df["daily_weight"]

first_prediction_date = pd.Timestamp(year=2025, month=1, day=1)
last_historical_date = pd.Timestamp(year=2024, month=12, day=20) #Last historical data ends at 19 december 2024. We might want to predict from this date then remove first entries.
last_prediction_date = pd.Timestamp(year=2025, month = 5, day = 31)

#We create the window for which we will make our predictions.
prediction_date_range = pd.date_range(start=last_historical_date, end=last_prediction_date)

#We only have to create one dataframe with the "common" variables, then we can predict in a loop and append those predictions to a new dataframe that should match the structure we want.

#We follow the same procedure as last section
#Add time-series data
prediction_map_df = pd.DataFrame({"date" : prediction_date_range})
prediction_map_df["day_of_week"] = prediction_map_df["date"].dt.dayofweek
prediction_map_df["is_weekend"]  = prediction_map_df['day_of_week'].isin([5, 6])
prediction_map_df["is_weekend"] = prediction_map_df["is_weekend"].astype(int)
prediction_map_df["month"] = prediction_map_df["date"].dt.month

#Add holidays
import holidays
years = list(range(2024, 2026))
all_holidays = holidays.Norway(years = years)
prediction_map_df["date_str"] = prediction_map_df["date"].dt.strftime('%Y-%m-%d')
holiday_str = set(pd.to_datetime(list(all_holidays.keys())).strftime('%Y-%m-%d'))
prediction_map_df["is_holiday"] = prediction_map_df["date_str"].isin(holiday_str)
prediction_map_df["is_holiday"] = prediction_map_df["is_holiday"].astype(int)

#Add empty columns for rolling windows
prediction_map_df["rolling_7"] = None
prediction_map_df["rolling_30"] = None
prediction_map_df["rolling_100"] = None

prediction_map_df.info()

valid_ids = final_df["rm_id"].unique()
for id in valid_ids:
    



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         163 non-null    datetime64[ns]
 1   day_of_week  163 non-null    int32         
 2   is_weekend   163 non-null    int64         
 3   month        163 non-null    int32         
 4   date_str     163 non-null    object        
 5   is_holiday   163 non-null    int64         
 6   rolling_7    0 non-null      object        
 7   rolling_30   0 non-null      object        
 8   rolling_100  0 non-null      object        
dtypes: datetime64[ns](1), int32(2), int64(2), object(4)
memory usage: 10.3+ KB


Run the code below to make predictions for the whole data. Note that this will override the all_predictions name from the test code.

In [ ]:
#For test_data we can use what we have.

all_predictions = []
valid_ids = final_df["rm_id"].unique()

for id, group in prediction_map_df.groupby("rm_id", sort=False):
    group = group.sort_values('date')
    X_group = group[xgb_X_columns]
    y_pred = xgb_model.predict(X_group)
    group["predicted_weight"] = y_pred
    group["rm_id"] = id
    all_predictions.append(group)


ValueError: Grouper and axis must be same length

In [ ]:
#Make csv to observe data
prediction_df.to_csv("test_predictions.csv", index=False)

In [ ]:
print(final_df["daily_weight"].mean())
print(final_df["daily_weight]"].var())

## 3.3.2 OPTION 2- USE ALL DATA
for actual submissions

### Preparing prediction window.
To make it easier when we make predictions, we will add all features we can to the prediction window.



